In [ ]:
import ee

# ==========================================
# 0. INITIALIZATION
# ==========================================

project_id = 'flood-forecasting-in-guinea'

try:
    ee.Initialize(project=project_id)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=project_id)

# ==========================================
# 1. DEFINE GUINEA ROI & 0.1° GRID (REDUCED)
# ==========================================
eraStep = 0.1
old_eraStep = 0.25
num_lon = 5
num_lat = 3
lon_min = -13.75
lat_min = 9.25

# Calculate original bounds and reduce by 6 columns (0.6 degrees)
original_lon_max = lon_min + (num_lon * old_eraStep) # -12.5
lat_max = lat_min + (num_lat * old_eraStep)         # 10.0
lon_max = original_lon_max - (6 * eraStep)          # -13.1

# Define the adjusted Bounding Box
roi = ee.Geometry.BBox(lon_min, lat_min, lon_max, lat_max)

# Define 0.1 degree resolution (approx 11,132 meters)
gridScale = 11132
projection = ee.Projection('EPSG:4326').atScale(gridScale)

# Generate the high-resolution grid
rawGrid = roi.coveringGrid(projection, gridScale)

# Assign clean IDs and extract Lat/Lon center for each cell
gridList = rawGrid.toList(rawGrid.size())

def prepare_grid(f):
    feature = ee.Feature(f)
    index = gridList.indexOf(f)
    centroid = feature.geometry().centroid(maxError=1).coordinates()
    return feature.set({
        'region_id': ee.Number(index).format('%d'),
        'lon': centroid.get(0),
        'lat': centroid.get(1)
    })

grid = ee.FeatureCollection(gridList.map(prepare_grid))


print(f"✅ Grid Created: {grid.size().getInfo()} Regions")

# ==========================================
# 2. DAILY RAINFALL AGGREGATION (mm)
# ==========================================
start_date = ee.Date('2001-01-01')
end_date = ee.Date('2024-12-31')

day_count = end_date.difference(start_date, 'days')
day_list = ee.List.sequence(0, day_count.subtract(1))

def aggregate_daily(day_offset):
    date = start_date.advance(day_offset, 'day')

    # Sum 24 hours and convert units
    daily_sum = ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY") \
        .filterDate(date, date.advance(1, 'day')) \
        .select('total_precipitation') \
        .sum() \
        .multiply(1000)

    return daily_sum.set({
        'date_str': date.format('YYYY-MM-dd'),
        'system:time_start': date.millis()
    })

daily_rainfall = ee.ImageCollection(day_list.map(aggregate_daily))

# ==========================================
# 3. SPATIAL REDUCTION
# ==========================================
def extract_stats(image):
    return image.reduceRegions(
        collection=grid,
        reducer=ee.Reducer.mean(),
        scale=gridScale
    ).map(lambda f: f.set('date', image.get('date_str')))

rainfall_table = daily_rainfall.map(extract_stats).flatten()

# Remove nulls (areas outside the boundary)
final_table = rainfall_table.filter(ee.Filter.notNull(['mean']))

# ==========================================
# 4. EXPORT TO DRIVE
# ==========================================
task = ee.batch.Export.table.toDrive(
    collection=final_table,
    description='Guinea_Rainfall_Daily_Reduced_Python',
    folder='GEE_Flood_Analysis',
    fileFormat='CSV',
    selectors=['region_id', 'date', 'lat', 'lon', 'mean']
)

task.start()

print("🚀 Task Submitted successfully! Monitor it in your GEE Task Manager or wait for the file in Google Drive.")


✅ Grid Created: 64 Regions
🚀 Task Submitted successfully! Monitor it in your GEE Task Manager or wait for the file in Google Drive.
